In [1]:
## Importing Libraries
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
import torch

In [2]:
torch.manual_seed(42)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'CPU')
print(f'Using Device : {device}')

Using Device : cuda


In [5]:
df = pd.read_csv('/content/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
df.shape

(60000, 785)

In [7]:
# train test split
from sklearn.model_selection import train_test_split
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# Scaling the features
X_train = X_train/255
X_test = X_test/255

In [9]:
## Creating CustomDataset Class
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype= torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [10]:
## Creating train dataset object
train_dataset = CustomDataset(X_train, y_train)
len(train_dataset)

48000

In [11]:
## Creating custom test dataset
test_dataset = CustomDataset(X_test, y_test)
len(test_dataset)

12000

In [12]:
## Creating train and test loader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)    # For accuracy

In [ ]:
## Creating Neural Network
class ANN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64, 10),
       # nn.Softmax()      # It is not be defined explicitly
    )

  def forward(self, features):
    return self.model(features)

In [ ]:
## Learning Rate and Epochs
learning_rate = 0.1
epochs = 100

In [ ]:
## Instanciate the Model
model = ANN(X_train.shape[1])
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [ ]:
## Training Loop
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_features, batch_labels in train_loader:

    # Move batches to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    ## Forward Pass
    outputs = model(batch_features)

    ## Calculate Loss
    loss = criterion(outputs, batch_labels)

    ## Backpropagation
    optimizer.zero_grad()
    loss.backward()

    ## Update Gradients
    optimizer.step()

    total_epoch_loss += loss.item()

  avg_loss = total_epoch_loss / len(train_loader)
  print(f'Epoch : {epoch}, Loss : {avg_loss}')

In [ ]:
## Model Evaluation
model.eval()

In [ ]:
## Accuracy on Test Data
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    # Move batches to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    outputs = model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total = total + batch_labels.shape[0]
    correct = correct + (predicted == batch_labels).sum().item()

  print(correct / total)

In [ ]:
## Accuracy on Training Data
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    # Move batches to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    outputs = model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total = total + batch_labels.shape[0]
    correct = correct + (predicted == batch_labels).sum().item()

  print(correct / total)